# 🏪 Laboratorio integrador: del dato a la decisión con el caso Mayoristas

## 📖 La historia

Un **distribuidor mayorista de alimentos en Portugal** vende a 440 clientes: hoteles, restaurantes y cafés (el canal **Horeca**) y tiendas de autoservicio y abarrotes (el canal **Retail**). Hoy los atiende a todos igual: el mismo vendedor, el mismo catálogo y el mismo camión.

El director comercial sospecha que eso le cuesta dinero: un restaurante que compra pescado fresco todos los días no necesita lo mismo que una tienda que compra cajas de detergente cada mes. Te entrega una tabla con el **gasto anual de cada cliente en seis categorías** y te pide una propuesta.

Este es el **cierre del curso**: vas a usar, en un solo proyecto, todo lo que aprendiste. Transformar y escalar (sesión 0), reducir con PCA (sesión 3), agrupar con K-Means (sesión 1), detectar atípicos con DBSCAN (sesión 2) y, sobre todo, **traducir los clusters a decisiones**.

> ### ❓ ¿Cómo debe organizar el distribuidor a sus clientes mayoristas: qué equipos de venta, qué surtido y qué logística le tocan a cada grupo?

## 🧭 Lo que vas a aprender

1. A reconocer datos de gasto **muy sesgados** y por qué conviene aplicar `log1p` **antes** de escalar.
2. A usar **PCA** como paso intermedio: resumir 6 categorías en 2 ejes que se pueden interpretar y graficar.
3. A combinar **K-Means** (segmentos) y **DBSCAN** (clientes atípicos) sobre el mismo espacio.
4. A **validar** un agrupamiento con una etiqueta que **no** usaste para agrupar (el canal de venta).
5. A convertir cada segmento en una **recomendación de negocio** con números en las unidades originales.

## ⏱️ Agenda (3 horas)

| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación, recap del curso y carga de datos | 10 min |
| 1 | Exploración: distribuciones sesgadas y outliers enormes | 20 min |
| 2 | ¿Por qué `log1p` antes de escalar? | 20 min |
| 3 | PCA: de 6 categorías a 2 ejes con significado | 25 min |
| ☕ | Descanso | 10 min |
| 4 | K-Means sobre los componentes principales | 25 min |
| 5 | DBSCAN: ¿quién no encaja en ningún segmento? | 20 min |
| 6 | Perfil de los segmentos en unidades originales | 20 min |
| 7 | Validación contra el canal de venta | 15 min |
| 8 | Recomendaciones de negocio y cierre | 15 min |
| | **Total** | **180 min** |

## 📋 Las variables

Datos: *Wholesale customers* (UCI Machine Learning Repository, Cardoso, 2014; licencia CC BY 4.0). Los gastos están en **unidades monetarias anuales** (u.m.).

| Variable (original) | En el notebook | Descripción | Unidades |
|---|---|---|---|
| `Channel` | `canal` | Horeca (hoteles, restaurantes, cafés) o Retail | categoría |
| `Region` | `region` | Lisboa, Oporto u Otra | categoría |
| `Fresh` | `frescos` | Gasto anual en productos frescos (frutas, verduras, carne, pescado) | u.m./año |
| `Milk` | `lacteos` | Gasto anual en lácteos | u.m./año |
| `Grocery` | `abarrotes` | Gasto anual en abarrotes (productos secos de anaquel) | u.m./año |
| `Frozen` | `congelados` | Gasto anual en congelados | u.m./año |
| `Detergents_Paper` | `detergentes_papel` | Gasto anual en detergentes y productos de papel | u.m./año |
| `Delicassen` | `delicatessen` | Gasto anual en delicatessen (embutidos, quesos finos) | u.m./año |

> ⚠️ **Regla del caso:** `canal` y `region` **no** entran al agrupamiento. Sólo agrupamos con los seis gastos. El canal lo guardamos para el final, como "examen" de nuestros segmentos.

---
# 0️⃣ Preparación (10 min)

## 📋 Recap: lo que ya sabes de las sesiones 1 a 3

| Sesión | Técnica | La idea clave | La trampa que aprendiste a evitar |
|---|---|---|---|
| 1 | **K-Means** | Asigna cada punto al centroide más cercano y recalcula; tú eliges `k` | Elegir `k` sólo con el codo; olvidar `n_init` y `random_state` |
| 2 | **DBSCAN** | Un cluster es una zona densa; lo que no cabe en ninguna es **ruido** (-1) | Elegir `eps` al azar en vez de usar la gráfica de k-distancia |
| 3 | **PCA** | Rota los datos para que pocos ejes (componentes) concentren la varianza | Aplicarlo sin escalar; no leer las cargas |
| Todas | **Escalado** | Las distancias y la varianza dependen de las unidades | Agrupar con variables en escalas distintas |

Hoy los encadenamos: **preparar → reducir → agrupar → validar → interpretar → actuar**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [2]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Cargar los datos

El archivo viene en la carpeta `datos/` junto a este notebook. Renombramos las columnas al español y traducimos los códigos de `canal` y `region` para leerlos más fácil.

In [3]:
# Esta celda va de regalo: léela con calma y ejecútala
import pandas as pd
import numpy as np

df = pd.read_csv('datos/wholesale_customers.csv')

df = df.rename(columns={
    'Channel': 'canal',
    'Region': 'region',
    'Fresh': 'frescos',
    'Milk': 'lacteos',
    'Grocery': 'abarrotes',
    'Frozen': 'congelados',
    'Detergents_Paper': 'detergentes_papel',
    'Delicassen': 'delicatessen',
})
df['canal'] = df['canal'].map({1: 'Horeca', 2: 'Retail'})
df['region'] = df['region'].map({1: 'Lisboa', 2: 'Oporto', 3: 'Otra'})

# Las seis variables de gasto: son las ÚNICAS que entran al agrupamiento
gasto = ['frescos', 'lacteos', 'abarrotes', 'congelados', 'detergentes_papel', 'delicatessen']

df.head()

,canal,region,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen
0,Retail,Otra,12669,9656,7561,214,2674,1338
1,Retail,Otra,7057,9810,9568,1762,3293,1776
2,Retail,Otra,6353,8808,7684,2405,3516,7844
3,Horeca,Otra,13265,1196,4221,6404,507,1788
4,Retail,Otra,22615,5410,7198,3915,1777,5185


---
# 1️⃣ Exploración: distribuciones sesgadas y outliers enormes (20 min)

Antes de agrupar hay que conocer los datos: cuántos son, si hay nulos, en qué rango se mueve cada gasto y cómo se relacionan entre sí.

In [4]:
# ¿Cuántas filas y columnas? ¿Hay valores nulos?
# Pista: df.shape y df.isnull().sum()
# ✍️ Tu código aquí
print(df.shape)
df.isnull().sum()

(440, 8)


canal                0
region               0
frescos              0
lacteos              0
abarrotes            0
congelados           0
detergentes_papel    0
delicatessen         0
dtype: int64

In [5]:
# Estadísticas descriptivas de los gastos
# Pista: df[gasto].describe().round(0)
# ✍️ Tu código aquí
df[gasto].describe().round(0)

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen
count,440.0,440.0,440.0,440.0,440.0,440.0
mean,12000.0,5796.0,7951.0,3072.0,2881.0,1525.0
std,12647.0,7380.0,9503.0,4855.0,4768.0,2820.0
min,3.0,55.0,3.0,25.0,3.0,3.0
25%,3128.0,1533.0,2153.0,742.0,257.0,408.0
50%,8504.0,3627.0,4756.0,1526.0,816.0,966.0
75%,16934.0,7190.0,10656.0,3554.0,3922.0,1820.0
max,112151.0,73498.0,92780.0,60869.0,40827.0,47943.0


In [6]:
# ¿Cuántos clientes hay por canal y por región?
# Pista: pd.crosstab(df['region'], df['canal'], margins=True)
# ✍️ Tu código aquí
pd.crosstab(df['region'], df['canal'], margins=True)

canal,Horeca,Retail,All
region,,,
Lisboa,59,18,77
Oporto,28,19,47
Otra,211,105,316
All,298,142,440


✍️ **¿Qué observas?** ¿Hay nulos? Compara la media y la mediana (50%) de cada gasto: ¿qué te dice que la media sea mucho mayor? ¿Cuánto vale el máximo comparado con la mediana? ¿Qué canal domina?

No hay nulos: son 440 clientes con 8 columnas completas. En todas las categorías la media es mucho mayor que la mediana (por ejemplo, frescos tiene media de 12,000 y mediana de 8,504; detergentes y papel tiene media de 2,881 y mediana de sólo 816). Eso dice que las distribuciones están sesgadas a la derecha: la mayoría compra poco y unos cuantos clientes muy grandes jalan el promedio hacia arriba. El máximo es enorme comparado con la mediana: en frescos es 112,151 contra 8,504 y en delicatessen 47,943 contra 966. El canal que domina es Horeca, con 298 de 440 clientes (contra 142 de Retail), y la mayoría de los clientes están en la región "Otra" (316).

In [7]:
# Histograma + boxplot de cada gasto
# Pista: plot_distributions(df, gasto)
# ✍️ Tu código aquí
plot_distributions(df, gasto)

In [8]:
# Esta celda va de regalo: léela con calma y ejecútala
# Dos números que resumen el sesgo de cada variable
sesgo = pd.DataFrame({
    'asimetria': df[gasto].skew().round(2),          # 0 = simétrica; > 1 = muy sesgada a la derecha
    'max_entre_mediana': (df[gasto].max() / df[gasto].median()).round(1),
})
sesgo

,asimetria,max_entre_mediana
frescos,2.56,13.2
lacteos,4.05,20.3
abarrotes,3.59,19.5
congelados,5.91,39.9
detergentes_papel,3.63,50.0
delicatessen,11.15,49.7


✍️ **¿Qué observas?** ¿Qué forma tienen los histogramas? ¿Qué variable es la más sesgada? ¿Cuántas veces la mediana gasta el cliente más grande? Si calculáramos distancias con estos datos, ¿qué clientes dominarían?

Todos los histogramas tienen la misma forma: una barra muy alta pegada a la izquierda y una cola larga hacia la derecha, con varios puntos sueltos en los boxplots. La más sesgada es delicatessen, con asimetría de 11.15. El cliente más grande gasta entre 13 y 50 veces lo que la mediana: 13.2 veces en frescos y hasta 50 veces en detergentes y papel y en delicatessen. Si calculáramos distancias así, dominarían esos pocos clientes gigantes: cualquier diferencia con ellos sería tan grande que el resto de los clientes quedarían todos amontonados y parecerían iguales.

In [9]:
# Matriz de correlación entre los gastos
# Pista: plot_correlation_matrix(df, gasto)
# ✍️ Tu código aquí
plot_correlation_matrix(df, gasto)

✍️ **¿Qué observas?** ¿Qué pares de variables están más correlacionados? ¿Ves un "bloque" de variables que suben juntas? ¿Qué tipo de negocio compraría esa canasta? ¿Por qué esto anticipa que PCA servirá?

El par más correlacionado es abarrotes con detergentes y papel (0.92), seguido de lácteos con abarrotes (0.73) y lácteos con detergentes y papel (0.66). Sí se ve un bloque claro de tres variables que suben juntas: lácteos, abarrotes y detergentes/papel. Esa canasta es la de una tienda o minisúper (retail), que vende productos empacados y de limpieza. Frescos y congelados van más por su lado (entre ellos 0.35) y se relacionan un poco con delicatessen. Esto anticipa que PCA va a servir porque esas tres variables cuentan casi la misma historia y se pueden resumir en un solo eje.

---
# 2️⃣ ¿Por qué `log1p` antes de escalar? (20 min)

`StandardScaler` resta la media y divide entre la desviación estándar, pero **no cambia la forma** de la distribución: un cliente que estaba 16 desviaciones arriba sigue igual de lejos. Para datos de dinero, el truco clásico es aplicar el **logaritmo** antes de escalar:

$$x' = \log(1 + x)$$

- El logaritmo **comprime la cola**: pasar de 1,000 a 10,000 pesa lo mismo que pasar de 10,000 a 100,000 (multiplicar por 10).
- Usamos `log1p` (es decir, $\log(1+x)$) en vez de `log` para que un gasto de 0 no dé $-\infty$.
- En lenguaje de negocio: comparamos **proporciones** ("gasta el doble") en vez de diferencias absolutas ("gasta 5,000 más").

Vamos a escalar **de las dos formas** y comparar.

In [10]:
# 1. Crea df_log aplicando np.log1p a df[gasto] (el df original NO se modifica)
# 2. Grafica sus distribuciones con plot_distributions(df_log, gasto)
# ✍️ Tu código aquí
df_log = np.log1p(df[gasto])
plot_distributions(df_log, gasto)

In [11]:
# Esta celda va de regalo: léela con calma y ejecútala
# Asimetría antes y después del logaritmo
pd.DataFrame({
    'asimetria_original': df[gasto].skew().round(2),
    'asimetria_log': df_log.skew().round(2),
})

,asimetria_original,asimetria_log
frescos,2.56,-1.58
lacteos,4.05,-0.22
abarrotes,3.59,-0.67
congelados,5.91,-0.35
detergentes_papel,3.63,-0.24
delicatessen,11.15,-1.09


✍️ **¿Qué observas?** ¿Cómo cambió la forma de los histogramas? Compara la asimetría antes y después. ¿Alguna variable quedó sesgada hacia el otro lado? ¿Qué clientes crees que están ahora en esa cola izquierda?

Los histogramas ya se ven mucho más parecidos a una campana, sin esa cola larga a la derecha. La asimetría bajó muchísimo: delicatessen pasó de 11.15 a -1.09, congelados de 5.91 a -0.35 y lácteos de 4.05 a -0.22. Pero algunas quedaron sesgadas hacia el otro lado, sobre todo frescos (-1.58) y delicatessen (-1.09). En esa cola izquierda ahora están los clientes que casi no compran esa categoría, con montos de 3 a 50 unidades, porque el logaritmo estira mucho las diferencias entre números chiquitos (por ejemplo 3 contra 300 ahora es una distancia grande).

In [12]:
from sklearn.preprocessing import StandardScaler

# 1. scaler_crudo = StandardScaler();  X_crudo = scaler_crudo.fit_transform(df[gasto])
# 2. scaler_log = StandardScaler();    X_log = scaler_log.fit_transform(df_log)
# 3. Imprime X_crudo.shape y X_log.shape
# ✍️ Tu código aquí
scaler_crudo = StandardScaler()
X_crudo = scaler_crudo.fit_transform(df[gasto])
scaler_log = StandardScaler()
X_log = scaler_log.fit_transform(df_log)
print(X_crudo.shape, X_log.shape)

(440, 6) (440, 6)


In [13]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Qué tan lejos queda el cliente más extremo de cada variable? (en desviaciones estándar)
pd.DataFrame({
    'max_|z|_crudo': np.abs(X_crudo).max(axis=0).round(1),
    'max_|z|_log': np.abs(X_log).max(axis=0).round(1),
    'celdas_|z|>3_crudo': (np.abs(X_crudo) > 3).sum(axis=0),
    'celdas_|z|>3_log': (np.abs(X_log) > 3).sum(axis=0),
}, index=gasto)

,max_|z|_crudo,max_|z|_log,celdas_|z|>3_crudo,celdas_|z|>3_log
frescos,7.9,5.0,7,6
lacteos,9.2,3.8,9,2
abarrotes,8.9,6.4,7,2
congelados,11.9,3.2,6,1
detergentes_papel,8.0,3.2,10,2
delicatessen,16.5,4.1,4,8


### 🧪 Prueba rápida: ¿cambia el agrupamiento?

Ajustamos un K-Means con k = 4 en cada espacio (un valor cualquiera, sólo para comparar; el k definitivo lo elegimos en la Parte 4) y vemos **cuántos clientes caen en cada cluster**.

In [14]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.cluster import KMeans

kmeans_crudo = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_crudo = kmeans_crudo.fit_predict(X_crudo)

print('Tamaños con datos crudos:', np.bincount(labels_crudo))
print('Silhouette (espacio crudo):', evaluar_clustering('crudo', X_crudo, labels_crudo)['silhouette'])

Tamaños con datos crudos: [110 315  10   5]
Silhouette (espacio crudo): 0.348


In [15]:
# Haz lo mismo con X_log:
# 1. kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
# 2. labels_log = kmeans_log.fit_predict(X_log)
# 3. Imprime los tamaños con np.bincount(labels_log)
# 4. Imprime la silhouette: evaluar_clustering('log', X_log, labels_log)['silhouette']
# ✍️ Tu código aquí
kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_log = kmeans_log.fit_predict(X_log)
print('Tamaños con log + escalado:', np.bincount(labels_log))
print('Silhouette (espacio log):', evaluar_clustering('log', X_log, labels_log)['silhouette'])

Tamaños con log + escalado: [ 61 129 132 118]
Silhouette (espacio log): 0.188


✍️ **¿Qué observas?** ¿Qué tan lejos queda el cliente más extremo en cada versión? Compara los tamaños de los clusters: ¿cuál te serviría para organizar equipos de venta? La silhouette del espacio crudo sale más alta: ¿qué está premiando? ¿Por qué no deberías compararla con la silhouette calculada en el espacio log?

En los datos crudos el cliente más extremo queda a 16.5 desviaciones estándar (en delicatessen) y en general entre 8 y 12; con logaritmo el máximo baja a 6.4 y casi todos quedan entre 3 y 5. Los tamaños con datos crudos son 110, 315, 10 y 5, o sea un cluster gigante y dos clusters de muy pocos clientes que seguramente son los compradores enormes. Con logaritmo salen 61, 129, 132 y 118, mucho más parejos. Para organizar equipos de venta me serviría el de logaritmo, porque no tiene sentido un equipo para 5 clientes y otro para 315. La silhouette del espacio crudo (0.348) sale más alta porque premia que unos pocos clientes estén lejísimos de todos: separarlos en clusters chiquitos hace que se vean muy "separados". No se puede comparar con la del espacio log (0.188) porque las distancias se miden en espacios distintos, con escalas distintas.

---
# 3️⃣ PCA: de 6 categorías a 2 ejes con significado (25 min)

Vimos que varias categorías se mueven juntas. PCA busca las direcciones que concentran la mayor varianza; si dos o tres ejes resumen casi todo, podemos **agrupar y graficar** en ese espacio reducido. Primero ajustamos PCA con **todos** los componentes para decidir cuántos quedarnos.

In [16]:
from sklearn.decomposition import PCA

# 1. pca_log = PCA() y ajústalo con pca_log.fit(X_log)
# 2. Imprime pca_log.explained_variance_ratio_ y su acumulada con np.cumsum(...)
# 3. Grafica con plot_varianza_explicada(pca_log)
# ✍️ Tu código aquí
pca_log = PCA()
pca_log.fit(X_log)
print('Varianza explicada:', pca_log.explained_variance_ratio_.round(3))
print('Acumulada:         ', np.cumsum(pca_log.explained_variance_ratio_).round(3))
plot_varianza_explicada(pca_log)

Varianza explicada: [0.441 0.272 0.107 0.101 0.049 0.03 ]
Acumulada:          [0.441 0.713 0.82  0.921 0.97  1.   ]


✍️ **¿Qué observas?** ¿Cuánta varianza explican PC1 y PC2 juntos? ¿Dónde ves el codo? ¿Cuántos componentes necesitarías para el 80%? ¿Con cuántos te quedarías y por qué?

PC1 y PC2 juntos explican 71.3% de la varianza (44.1% y 27.2%). El codo se ve en PC3: de PC2 a PC3 la varianza cae de 27.2% a 10.7% y después PC3 y PC4 casi empatan (10.7% y 10.1%). Para el 80% necesitaría 3 componentes (82%). Me quedo con 2 componentes, porque están antes del codo, conservan más de 70% de la información, se pueden graficar directamente y, como se ve en las cargas, cada uno tiene un significado de negocio claro.

In [17]:
# ¿Qué variables pesan en cada componente?
# Pista: plot_cargas(pca_log, gasto, n_componentes=3)
# ✍️ Tu código aquí
plot_cargas(pca_log, gasto, n_componentes=3)

✍️ **¿Qué observas?** Completa la tabla con las variables que más pesan en cada componente y ponle un nombre de negocio a cada eje.

| Componente | Variables con más peso | Nombre de negocio |
|---|---|---|
| PC1 | abarrotes (0.57), detergentes_papel (0.55), lacteos (0.54) | Canasta de tienda / retail |
| PC2 | frescos (0.59), congelados (0.59), delicatessen (0.53) | Canasta de cocina / restaurante (Horeca) |

PC1 junta justo el bloque de la matriz de correlación (abarrotes, detergentes y papel, lácteos): un cliente con PC1 alto compra mucho producto empacado y de limpieza, como una tienda. PC2 junta frescos, congelados y delicatessen, que es lo que compra una cocina: un cliente con PC2 alto compra mucho producto para preparar comida, como un restaurante u hotel. Las dos dimensiones son casi independientes entre sí.

In [18]:
# 1. pca_final = PCA(n_components=<los componentes que elegiste>)
# 2. X_pcs = pca_final.fit_transform(X_log)
# 3. df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
# 4. Grafica con plot_biplot(X_pcs, pca_final, gasto)   (dibuja PC1 contra PC2)
# ✍️ Tu código aquí
pca_final = PCA(n_components=2)
X_pcs = pca_final.fit_transform(X_log)
df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
plot_biplot(X_pcs, pca_final, gasto)

✍️ **¿Qué observas?** ¿Hacia dónde apuntan las flechas de cada canasta? ¿Ves grupos claramente separados en la nube o una sola masa? ¿Hay clientes lejos del resto? ¿Qué significa estar arriba a la derecha del biplot?

Las flechas de lácteos, abarrotes y detergentes/papel apuntan hacia la derecha (PC1), mientras que las de frescos, congelados y delicatessen apuntan hacia arriba (PC2). La nube no tiene grupos claramente separados, se ve más como una sola masa con dos "lóbulos": uno a la izquierda y otro a la derecha. Sí hay algunos clientes lejos del resto, sobre todo abajo a la izquierda y en las orillas. Estar arriba a la derecha significa comprar mucho de las dos canastas a la vez: son clientes grandes que compran de todo.

---
# ☕ Descanso (10 min)

---
# 4️⃣ K-Means sobre los componentes principales (25 min)

Ahora sí, agrupamos en el espacio `X_pcs` (los componentes que elegiste). Como en la sesión 1, elegimos `k` con **varias señales**: codo, silhouette y gráfica de silueta... y con **sentido de negocio**.

In [19]:
# Inercia y silhouette de K-Means para k = 2 ... 8 en el espacio de componentes
# Pista: tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
# ✍️ Tu código aquí
tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
tabla_k

,k,inercia,silhouette
0,2,1086.1,0.408
1,3,796.6,0.392
2,4,632.8,0.329
3,5,531.7,0.340
4,6,444.0,0.352
5,7,379.4,0.362
6,8,333.3,0.357


In [20]:
# 1. Grafica el codo: plot_codo(tabla_k['k'], tabla_k['inercia'])
# 2. Grafica la silhouette: plot_silueta_k(tabla_k)
# ✍️ Tu código aquí
plot_codo(tabla_k['k'], tabla_k['inercia'])
plot_silueta_k(tabla_k)

✍️ **¿Qué observas?** ¿Dónde ves el codo? ¿Qué k prefiere la silhouette? ¿Coinciden? Si los candidatos casi empatan, ¿qué criterio usarías para decidir? Escribe el k que eliges y por qué.

El codo no es muy claro, pero se ve alrededor de k = 3 o 4: la inercia baja de 1,086 (k=2) a 797 (k=3) y 633 (k=4), y después baja más despacio. La silhouette prefiere k = 2 (0.408), pero k = 3 está casi empatado (0.392). Como casi empatan, usaría el criterio de negocio: con 3 grupos se distinguen los clientes tipo tienda, los tipo restaurante y además los clientes grandes que compran de todo, lo que da 3 equipos de venta con un perfil distinto cada uno. Elijo k = 3.

In [21]:
# 1. kmeans_final = KMeans(n_clusters=<tu k>, n_init=10, random_state=42)
# 2. labels_final = kmeans_final.fit_predict(X_pcs)
# 3. Imprime los tamaños con np.bincount(labels_final)
# 4. plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_)
# ✍️ Tu código aquí
kmeans_final = KMeans(n_clusters=3, n_init=10, random_state=42)
labels_final = kmeans_final.fit_predict(X_pcs)
print('Tamaños:', np.bincount(labels_final))
plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_)

Tamaños: [ 83 212 145]


In [22]:
# Gráfica de silueta de tu modelo
# Pista: plot_silueta(X_pcs, labels_final)
# ✍️ Tu código aquí
plot_silueta(X_pcs, labels_final)

In [23]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Perdimos algo por agrupar en los componentes principales y no en las 6 variables (log + escaladas)?
kmeans_6d = KMeans(n_clusters=kmeans_final.n_clusters, n_init=10, random_state=42)
labels_6d = kmeans_6d.fit_predict(X_log)

# Tabla cruzada: filas = grupo usando los PCs, columnas = grupo usando las 6 variables
# (los números de cluster son arbitrarios: el "0" de un modelo puede ser el "1" del otro)
tabla_pcs_vs_6d = pd.crosstab(
    pd.Series(labels_final, name='K-Means en PCs'),
    pd.Series(labels_6d, name='K-Means en 6 variables'),
)
# Para cada grupo de los PCs, la celda más grande son los clientes que "se quedan juntos"
print('Clientes que quedan en el mismo grupo:', tabla_pcs_vs_6d.max(axis=1).sum(), 'de', len(labels_final))
tabla_pcs_vs_6d

Clientes que quedan en el mismo grupo: 437 de 440


K-Means en 6 variables,0,1,2
K-Means en PCs,,,
0,80,2,1
1,0,0,212
2,0,145,0


✍️ **¿Qué observas?** ¿Cuántos clientes quedaron en cada cluster? ¿Dónde está cada cluster en el plano PC1-PC2 y qué significa eso en términos de canastas? ¿La silueta es alta o moderada? ¿Por qué? ¿Perdimos información al agrupar con los componentes en vez de las 6 variables?

Quedaron 83 clientes en el cluster 0, 212 en el cluster 1 y 145 en el cluster 2. El cluster 0 está a la derecha y abajo (PC1 alto, PC2 bajo): compra la canasta de tienda pero casi nada de frescos y congelados. El cluster 1 está a la izquierda (PC1 bajo): compra la canasta de cocina y casi nada de abarrotes y detergentes. El cluster 2 está a la derecha y arriba (PC1 y PC2 altos): compra mucho de todo. La silueta es moderada (0.392 de promedio; 0.45 en el cluster 1 y alrededor de 0.33 en los otros dos) porque, como se vio en el biplot, los clientes forman más un continuo que grupos totalmente separados, y hay clientes en las fronteras. Casi no perdimos información: 437 de 440 clientes quedan en el mismo grupo si agrupamos con las 6 variables, sólo cambian 3.

### 📊 Guardamos los resultados

In [24]:
# 1. comparacion = []
# 2. comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))
# ✍️ Tu código aquí
comparacion = []
comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))
pd.DataFrame(comparacion)

,modelo,n_clusters,pct_ruido,silhouette
0,K-Means final (PCs),3,0.0,0.392


---
# 5️⃣ DBSCAN: ¿quién no encaja en ningún segmento? (20 min)

K-Means **obliga** a cada cliente a pertenecer a un cluster, aunque esté lejísimos de todos. DBSCAN no: lo que no está en una zona densa queda como **ruido**. Lo usamos sobre el **mismo espacio** `X_pcs` para encontrar clientes atípicos.

Receta de la sesión 2:
1. `min_samples` ≈ 2 × número de dimensiones. Calcúlalo con los componentes que elegiste: `min_samples = 2 * X_pcs.shape[1]`.
2. `eps` sale de la **gráfica de k-distancia** con k = `min_samples`: buscamos el codo donde la curva se dispara.

In [25]:
# 1. min_samples = 2 * X_pcs.shape[1]
# 2. Grafica la k-distancia: plot_k_distancia(X_pcs, k=min_samples)
# 3. Ubica el codo (donde la curva se dispara) y vuelve a graficar con eps=<tu eps> para verificar
# ✍️ Tu código aquí
min_samples = 2 * X_pcs.shape[1]
print('min_samples =', min_samples)
plot_k_distancia(X_pcs, k=min_samples)
plot_k_distancia(X_pcs, k=min_samples, eps=0.8)

from sklearn.neighbors import NearestNeighbors
dist_k = NearestNeighbors(n_neighbors=min_samples).fit(X_pcs).kneighbors(X_pcs)[0][:, -1]
print(f'% de clientes por encima de eps = 0.8: {100 * (dist_k > 0.8).mean():.1f}%')

min_samples = 4


% de clientes por encima de eps = 0.8: 5.9%


✍️ **¿Qué observas?** ¿En qué valor de la distancia empieza a dispararse la curva? ¿Qué `eps` elegiste? ¿Qué porcentaje aproximado de clientes queda por encima de tu línea?

La curva es plana y baja hasta un poco después de 0.5 y empieza a dispararse alrededor de 0.8; arriba de 1 sube muy rápido. Elegí eps = 0.8, justo donde empieza el codo. Por encima de la línea queda más o menos el 6% de los clientes (5.9%).

In [26]:
from sklearn.cluster import DBSCAN

# 1. dbscan_atipicos = DBSCAN(eps=<tu eps>, min_samples=min_samples)
# 2. labels_db = dbscan_atipicos.fit_predict(X_pcs)
# 3. fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
# 4. comparacion.append(fila_db) y muestra fila_db
# ✍️ Tu código aquí
dbscan_atipicos = DBSCAN(eps=0.8, min_samples=min_samples)
labels_db = dbscan_atipicos.fit_predict(X_pcs)
fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
comparacion.append(fila_db)
fila_db

{'modelo': 'DBSCAN (PCs)',
 'n_clusters': 1,
 'pct_ruido': 4.5,
 'silhouette': nan}

In [27]:
# Grafica el resultado de DBSCAN (el ruido sale en gris)
# Pista: plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')
# ✍️ Tu código aquí
plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')

In [28]:
# Esta celda va de regalo: léela con calma y ejecútala
# Sensibilidad: ¿qué pasa con otros valores de eps?
pd.DataFrame([
    evaluar_clustering(f'eps={eps}', X_pcs, DBSCAN(eps=eps, min_samples=dbscan_atipicos.min_samples).fit_predict(X_pcs))
    for eps in [0.4, 0.6, 0.8, 1.0, 1.2]
])

,modelo,n_clusters,pct_ruido,silhouette
0,eps=0.4,4,10.0,0.080
1,eps=0.6,2,7.0,0.344
2,eps=0.8,1,4.5,NaN
3,eps=1.0,2,1.8,0.420
4,eps=1.2,1,1.1,NaN


✍️ **¿Qué observas?** ¿Cuántos clusters encontró DBSCAN y qué porcentaje es ruido? ¿Por qué la silhouette sale `NaN`? ¿Qué te dice esto sobre si los clientes forman grupos separados o un continuo? Según la tabla de sensibilidad, ¿qué tan estable es el porcentaje de ruido?

DBSCAN encontró 1 solo cluster y 4.5% de ruido (20 clientes). La silhouette sale NaN porque se necesita al menos 2 clusters para calcularla; con uno solo no hay "cluster vecino" con el que comparar. Esto dice que los clientes no forman grupos separados por zonas vacías, sino un continuo denso: los segmentos de K-Means son cortes que hacemos nosotros dentro de esa masa, útiles para el negocio pero sin fronteras naturales. En la tabla de sensibilidad el ruido es bastante estable entre 0.6 y 1.0 (de 7% a 1.8%) y siempre es una minoría, aunque el número de clusters sí brinca entre 1 y 2 según eps.

### 🔎 ¿Quiénes son los atípicos?

Miramos a los clientes marcados como ruido **en unidades originales** (u.m. al año), ordenados por su gasto total.

In [29]:
# Esta celda va de regalo: léela con calma y ejecútala
df['gasto_total'] = df[gasto].sum(axis=1)

atipicos = df[labels_db == -1].sort_values('gasto_total', ascending=False)
atipicos[gasto + ['gasto_total', 'canal']]

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,canal
85,16117,46197,92780,1026,40827,2944,199891,Retail
47,44466,54259,55571,7782,24171,6465,192714,Retail
181,112151,29627,18148,16745,4948,8550,190169,Horeca
183,36847,43950,20170,36534,239,47943,185683,Horeca
325,32717,16784,13626,60869,1272,5609,130877,Horeca
23,26373,36423,22019,5154,4337,16523,110829,Retail
65,85,20959,45828,36,24231,1423,92562,Retail
109,1406,16729,28986,673,836,3,48633,Retail
289,42786,286,471,1388,32,22,44985,Horeca
357,37,1275,22272,137,6747,110,30578,Retail


✍️ **¿Qué observas?** Mira a los clientes atípicos: ¿todos son grandes compradores? ¿Ves algún patrón distinto entre ellos? ¿Qué tipos de atípicos distingues? ¿Por qué el logaritmo hizo visibles a algunos de ellos?

No todos son grandes compradores. Distingo dos tipos: arriba hay clientes enormes, con gasto total de 110 mil a 200 mil al año (por ejemplo, el 85 y el 47 son Retail con muchísimo de abarrotes y detergentes, y el 181 y el 183 son Horeca con muchísimo de frescos o delicatessen). Abajo hay clientes con canastas muy raras: compran casi cero en varias categorías, como el 75 que tiene 3 de abarrotes y 3 de detergentes, el 289 que compra mucho de frescos pero casi nada de lo demás, o el 154 que gasta sólo 904 en total. El logaritmo hizo visibles a este segundo grupo: con los datos crudos un gasto de 3 y uno de 300 se ven casi iguales frente a los miles de los demás, pero en logaritmo esa diferencia es enorme y esos clientes quedan lejos de todos.

---
# 6️⃣ Perfil de los segmentos en unidades originales (20 min)

Un cluster sólo es útil si se puede **explicar en el idioma del negocio**. Nadie en el área comercial entiende "PC1 = 1.18": hay que regresar a **u.m. al año**. Por eso aplicamos `perfil_clusters` sobre el `df` **original**, sin logaritmo ni escalado.

In [30]:
# Perfil de cada cluster en unidades originales (u.m. al año)
# Pista: perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
# ✍️ Tu código aquí
perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
perfil_final

,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Cluster 0,83,18.9,2966.9,7010.9,12397.3,608.3,5446.2,795.0,29224.6
Cluster 1,212,48.2,11992.6,1995.4,2496.9,3277.8,425.9,892.5,21081.2
Cluster 2,145,33.0,17182.4,10658.1,13381.0,4181.1,5003.6,2867.2,53273.4


In [31]:
# Esta celda va de regalo: léela con calma y ejecútala
# Como los gastos son muy sesgados, la mediana describe mejor al cliente típico de cada cluster
medianas = df[gasto + ['gasto_total']].groupby(labels_final).median().round(0)
medianas.index = [f'Cluster {c}' for c in medianas.index]

# ¿Qué porcentaje de la venta total aporta cada cluster?
medianas['pct_de_la_venta'] = (
    100 * df.groupby(labels_final)['gasto_total'].sum() / df['gasto_total'].sum()
).round(1).values
medianas

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,pct_de_la_venta
Cluster 0,1531.0,6154.0,10487.0,398.0,4196.0,490.0,27251.0,16.6
Cluster 1,9635.0,1598.0,2151.0,2151.0,274.0,686.0,18498.0,30.6
Cluster 2,12205.0,7209.0,9965.0,2033.0,3378.0,2005.0,46100.0,52.8


✍️ **¿Qué observas?** Para cada cluster: ¿en qué categorías gasta más y en cuáles casi nada? ¿Cuál es el más grande en número de clientes? ¿Cuál aporta más venta? Ponle un **nombre de negocio** a cada segmento.

| Cluster | Nombre del segmento | ¿Qué lo distingue? |
|---|---|---|
| 0 | Tiendas de abarrotes | 83 clientes (16.6% de la venta). Mediana alta en abarrotes (10,487), lácteos (6,154) y detergentes (4,196); casi nada de frescos (1,531) y congelados (398) |
| 1 | Restaurantes y cafés | 212 clientes, el más grande (30.6% de la venta). Compran sobre todo frescos (9,635) y congelados (2,151); casi nada de detergentes (274) y poco de abarrotes y lácteos |
| 2 | Clientes grandes de canasta completa | 145 clientes y el que más vende (52.8%). Gastan mucho en todo: frescos (12,205), abarrotes (9,965), lácteos (7,209) y el más alto en delicatessen (2,005). Gasto total mediano de 46,100 |

(Agrega una fila por cada cluster de tu modelo.)

El más grande en número de clientes es el de restaurantes y cafés (212, casi la mitad), pero el que aporta más venta es el de clientes grandes de canasta completa: con un tercio de los clientes trae más de la mitad de la venta (52.8%). Las tiendas de abarrotes son el segmento más chico y el que menos vende.

In [32]:
# Perfil del ruido de DBSCAN contra el cluster denso
# Pista: perfil_clusters(df, labels_db, gasto + ['gasto_total'])
# ✍️ Tu código aquí
perfil_clusters(df, labels_db, gasto + ['gasto_total'])

,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Ruido,20,4.5,16775.5,14424.4,18206.0,6878.4,5984.6,4590.0,66858.8
Cluster 0,420,95.5,11772.9,5385.4,7463.0,2890.7,2733.7,1378.9,31624.6


In [33]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿En qué segmento de K-Means había quedado cada atípico?
pd.crosstab(
    pd.Series(labels_db, name='DBSCAN').map(lambda c: 'Ruido' if c == -1 else 'Denso'),
    pd.Series(labels_final, name='K-Means')
)

K-Means,0,1,2
DBSCAN,,,
Denso,74,207,139
Ruido,9,5,6


✍️ **¿Qué observas?** ¿El promedio del ruido describe bien a los atípicos? ¿Por qué no? ¿En qué clusters de K-Means habían quedado? ¿Qué pasaría si no los separáramos?

No, el promedio del ruido no describe bien a los atípicos. Sale como un cliente que gasta 66,859 en total, más del doble que el cluster denso, pero eso es porque mezcla a clientes gigantes (de casi 200 mil) con clientes que casi no compran nada (de menos de mil). El promedio queda en medio y no se parece a ninguno. Estaban repartidos en los 3 clusters de K-Means: 9 en tiendas, 5 en restaurantes y 6 en canasta completa. Si no los separáramos, los gigantes inflarían los promedios de su segmento y los que casi no compran quedarían como si fueran clientes normales, así que el perfil de cada segmento y las decisiones de surtido y logística quedarían distorsionados.

---
# 7️⃣ Validación contra el canal de venta (15 min)

Ahora sí sacamos la variable que guardamos: `canal` (Horeca o Retail). **No la usamos para agrupar**, así que es una prueba honesta: si nuestros segmentos tienen sentido, deberían relacionarse con el tipo de negocio del cliente.

⚠️ En la mayoría de los proyectos reales **no tienes** una etiqueta así. Aquí tenemos la suerte de poder revisar; aun así, no buscamos que los clusters **copien** el canal: buscamos que lo **expliquen** y, si se puede, que digan algo más.

In [34]:
# Tabla cruzada: canal (filas) contra tus clusters (columnas)
# Pista: pd.crosstab(df['canal'], labels_final, rownames=['canal'], colnames=['cluster'], margins=True)
# ✍️ Tu código aquí
pd.crosstab(df['canal'], labels_final, rownames=['canal'], colnames=['cluster'], margins=True)

cluster,0,1,2,All
canal,,,,
Horeca,33,209,56,298
Retail,50,3,89,142
All,83,212,145,440


In [35]:
# Channel tiene 2 categorías: ajusta también un K-Means con k = 2 para compararlo de tú a tú
# 1. kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
# 2. labels_k2 = kmeans_k2.fit_predict(X_pcs)
# 3. comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
# 4. Tabla cruzada: pd.crosstab(df['canal'], labels_k2, rownames=['canal'], colnames=['cluster'], margins=True)
# ✍️ Tu código aquí
kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_k2 = kmeans_k2.fit_predict(X_pcs)
comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
pd.crosstab(df['canal'], labels_k2, rownames=['canal'], colnames=['cluster'], margins=True)

cluster,0,1,All
canal,,,
Horeca,54,244,298
Retail,134,8,142
All,188,252,440


In [36]:
# Esta celda va de regalo: léela con calma y ejecútala
# Mismas tablas, pero en % de cada cluster (cada columna suma 100)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    print(f'--- {nombre}: % de cada cluster por canal ---')
    print((100 * pd.crosstab(df['canal'], labels, rownames=['canal'], colnames=['cluster'], normalize='columns')).round(1))

# ¿Cuántos clientes quedan en un cluster cuyo canal mayoritario NO es el suyo?
# (en cada columna, todo lo que no está en la celda más grande)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    tabla = pd.crosstab(df['canal'], labels)
    print(f'{nombre}: {len(labels) - tabla.max(axis=0).sum()} de {len(labels)} clientes fuera del canal mayoritario de su cluster')

--- K-Means final: % de cada cluster por canal ---
cluster     0     1     2
canal                    
Horeca   39.8  98.6  38.6
Retail   60.2   1.4  61.4
--- K-Means k=2: % de cada cluster por canal ---
cluster     0     1
canal              
Horeca   28.7  96.8
Retail   71.3   3.2
K-Means final: 92 de 440 clientes fuera del canal mayoritario de su cluster
K-Means k=2: 62 de 440 clientes fuera del canal mayoritario de su cluster


In [37]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Y la región? Comparamos la mezcla de regiones de cada cluster con la del total
mezcla_region = 100 * pd.crosstab(df['region'], labels_final, rownames=['region'], colnames=['cluster'], normalize='columns')
mezcla_region['Total'] = 100 * df['region'].value_counts(normalize=True)
mezcla_region.round(1)

cluster,0,1,2,Total
region,,,,
Lisboa,16.9,18.9,15.9,17.5
Oporto,12.0,12.3,7.6,10.7
Otra,71.1,68.9,76.6,71.8


In [38]:
# Dibuja el biplot coloreado por el canal real
# Pista: plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')
# ✍️ Tu código aquí
plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')

✍️ **¿Qué observas?** ¿Algún cluster es casi puro de un canal? ¿Qué % de cada uno de tus segmentos es Retail? ¿Cuál de los modelos se parece más al canal (cuántos clientes quedan fuera del canal mayoritario de su cluster)? Si tu modelo final se parece menos al canal, ¿significa que está mal? ¿La mezcla de regiones cambia de un cluster a otro? ¿Qué te dice eso sobre organizar a los vendedores por zona? En el biplot, ¿hacia dónde se cargan los Retail y los Horeca?

El cluster 1 (restaurantes y cafés) es casi puro Horeca: 98.6%, sólo 3 de 212 clientes son Retail. Las tiendas de abarrotes son 60.2% Retail y los clientes de canasta completa 61.4% Retail. El modelo de k = 2 se parece más al canal: deja 62 clientes fuera del canal mayoritario de su cluster contra 92 del modelo final. Pero eso no significa que el modelo final esté mal: no buscábamos copiar el canal, y el modelo de 3 grupos dice algo que el canal no dice, que entre los Retail y algunos Horeca hay clientes chicos tipo tienda y clientes grandes que compran de todo (este último grupo trae más de la mitad de la venta). La mezcla de regiones casi no cambia entre clusters: en todos anda alrededor de 17% Lisboa, 10% Oporto y 70% Otra, igual que el total. Eso dice que organizar a los vendedores sólo por zona no separaría los tipos de clientes; cada zona tiene de todos. En el biplot los Retail se cargan hacia la derecha (canasta de tienda, PC1 alto) y los Horeca hacia la izquierda y arriba (canasta de cocina).

---
# 8️⃣ Recomendaciones de negocio y cierre (15 min)

El último paso del flujo es **actuar**. Para cada segmento decide: qué **equipo de venta** lo atiende, qué **surtido** se le ofrece y qué **logística** necesita. Y no olvides a los atípicos.

### 💼 Recomendaciones por segmento

✍️ Completa la tabla con **tus** segmentos (los nombres que elegiste en la Parte 6) y tus números.

| Segmento | Clientes | % de la venta | Equipo de venta | Surtido | Logística |
|---|---|---|---|---|---|
| Clientes grandes de canasta completa (2) | 145 | 52.8% | Ejecutivos de cuenta clave (key account), cada uno con pocos clientes | Catálogo completo, con énfasis en delicatessen y productos premium; precios por volumen | Rutas dedicadas y entregas frecuentes, 2 a 3 veces por semana, con camión grande |
| Restaurantes y cafés (1) | 212 | 30.6% | Equipo Horeca, con vendedores que conocen cocina y pedidos por teléfono o app | Frescos y congelados, con algo de delicatessen | Entregas diarias o muy frecuentes en la mañana, con cadena de frío |
| Tiendas de abarrotes (0) | 83 | 16.6% | Equipo de retail / preventa que visita tiendas | Abarrotes, lácteos y detergentes/papel, con promociones y exhibición | Entregas semanales consolidadas por ruta; no necesitan refrigeración especial |
| Atípicos (DBSCAN) | 20 | — | Revisión caso por caso: los gigantes con ejecutivo key account; los de compra mínima con televenta | A la medida de su canasta | Según el cliente: ruta dedicada para los grandes, consolidar con otras rutas para los chicos |

(Agrega una fila por cada segmento de tu modelo.)

✍️ ¿Qué harías con los clientes atípicos? ¿Les darías el mismo trato a todos?

No les daría el mismo trato a todos. A los clientes gigantes (como el 85, 47, 181 y 183, de casi 200 mil al año) los atendería de forma personalizada con un ejecutivo de cuenta clave, porque perder a uno de ellos pega fuerte en la venta. A los que tienen canastas raras o compran casi nada (como el 154 o el 75) primero revisaría si hay errores de captura o si sólo compran una categoría, y si son reales los atendería por televenta o con pedidos mínimos, sin gastar en visitas.

### 📊 Tabla comparativa final

In [39]:
# 1. tabla_comparacion = pd.DataFrame(comparacion)
# 2. Muestra tabla_comparacion
# ✍️ Tu código aquí
tabla_comparacion = pd.DataFrame(comparacion)
tabla_comparacion

,modelo,n_clusters,pct_ruido,silhouette
0,K-Means final (PCs),3,0.0,0.392
1,DBSCAN (PCs),1,4.5,NaN
2,K-Means k=2 (PCs),2,0.0,0.408


✍️ **¿Qué observas?** ¿Cuál modelo gana en cada métrica? ¿Por qué sí podemos comparar estas silhouettes entre sí (a diferencia de la del espacio crudo)? ¿Qué aporta DBSCAN si no tiene silhouette?

En silhouette gana K-Means con k = 2 (0.408), aunque el modelo final con k = 3 está muy cerca (0.392). DBSCAN no tiene silhouette porque encontró un solo cluster, pero es el único que detecta atípicos (4.5%). Estas silhouettes sí se pueden comparar porque las tres se calcularon en el mismo espacio, X_pcs (log, escalado y PCA de 2 componentes); la del espacio crudo estaba en otras unidades y premiaba aislar a los clientes gigantes. Lo que aporta DBSCAN es otra cosa: decirnos que los clientes forman un continuo y señalar a los 20 clientes que no se parecen a nadie, que son justo los que hay que tratar aparte.

## ❓ Respuesta a la pregunta

> **¿Cómo debe organizar el distribuidor a sus clientes mayoristas?**

✍️ Escribe tu respuesta en 4-6 líneas: cuántos segmentos, cómo se llaman, cuántos clientes y qué % de la venta trae cada uno, qué equipo/surtido/logística le toca, qué hacer con los atípicos y si la región o el canal bastarían para organizarlos.

El distribuidor debería organizar a sus clientes en 3 segmentos según lo que compran: clientes grandes de canasta completa (145 clientes, 52.8% de la venta), atendidos por ejecutivos de cuenta clave con catálogo completo y rutas dedicadas; restaurantes y cafés (212 clientes, 30.6%), con un equipo Horeca enfocado en frescos y congelados y entregas frecuentes con frío; y tiendas de abarrotes (83 clientes, 16.6%), con preventa de abarrotes, lácteos y limpieza en entregas semanales. Los 20 atípicos que marca DBSCAN se revisan uno por uno: los gigantes con atención personalizada y los de compra mínima por televenta. La región no sirve para organizarlos porque todas las zonas tienen la misma mezcla, y el canal ayuda pero no basta: dentro de Retail y Horeca hay clientes grandes que compran de todo y que son los que más venden.

## 📝 Lo que aprendimos

✍️ Escribe las 3 a 5 lecciones más importantes de este caso (y del curso).

1. Antes de agrupar hay que ver la forma de los datos: con montos de dinero muy sesgados, log1p y después StandardScaler hace que los clientes gigantes no dominen las distancias.
2. PCA no sólo reduce columnas: con las cargas se le puede poner nombre de negocio a cada eje (canasta de tienda y canasta de cocina) y agrupar en 2 dimensiones casi no cambió los resultados (437 de 440 iguales).
3. Para elegir k se combinan métricas y sentido de negocio; cuando la silhouette casi empata, gana el número de grupos que se puede convertir en equipos y acciones.
4. K-Means y DBSCAN se complementan: K-Means segmenta a todos y DBSCAN señala a los atípicos que distorsionan los promedios.
5. Los perfiles se presentan en unidades originales (y mejor con medianas), y las silhouettes sólo se comparan si están calculadas en el mismo espacio.

## 🚀 Retos opcionales

1. Repite el análisis con **un componente más** de los que elegiste. ¿Cambian los segmentos?
2. Prueba **K-Means con otro k**. ¿El segmento nuevo es accionable o sólo parte uno existente?
3. Usa **HDBSCAN** (`sklearn.cluster.HDBSCAN`) sobre `X_pcs`. ¿Encuentra los mismos atípicos?
4. Agrupa con **proporciones** en vez de montos: divide cada gasto entre el `gasto_total` del cliente. ¿Aparecen segmentos por *tipo* de canasta sin importar el tamaño?